In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [40]:
pd.read_csv('../logs/events_large_demo.csv').head(3)

,ocel:eid,ocel:activity,ocel:timestamp,lifecycle,dur_s,lot_id,slot_no,eqp_id,module_id,srcmoduleid,dstmoduleid,state,area
0,e1,PURGE_START,2026-01-01 08:00:00,start,0.0,NaN,NaN,EQP_LP,LP3,NaN,NaN,PURGE,LP
1,e2,PURGE_END,2026-01-01 08:05:42,complete,342.0,NaN,NaN,EQP_LP,LP3,NaN,NaN,PURGE,LP
2,e3,PURGE_START,2026-01-01 08:06:00,start,0.0,NaN,NaN,EQP_LP,LP3,NaN,NaN,PURGE,LP


In [43]:
# =========================================================
# 0. 경로 설정
# =========================================================
BASE_DIR = Path("../logs/")  # 폴더 위치에 맞게 수정
RAW_PATH = BASE_DIR / "raw_df_large_demo.csv"
EVENTS_PATH = BASE_DIR / "events_large_demo.csv"
OBJECTS_PATH = BASE_DIR / "objects_large_demo.csv"
REL_EXT_PATH = BASE_DIR / "relations_ext_large_demo.csv"
SUMMARY_PATH = BASE_DIR / "summary.json"

print(BASE_DIR)
print(RAW_PATH)
print(EVENTS_PATH)
print(OBJECTS_PATH)
print(REL_EXT_PATH)
print(SUMMARY_PATH)

..\logs
..\logs\raw_df_large_demo.csv
..\logs\events_large_demo.csv
..\logs\objects_large_demo.csv
..\logs\relations_ext_large_demo.csv
..\logs\summary.json


In [47]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. 경로 설정
# =========================================================
BASE_DIR = Path("..logs")  # 폴더 위치에 맞게 수정
RAW_PATH = BASE_DIR / "raw_df_large_demo.csv"
EVENTS_PATH = BASE_DIR / "events_large_demo.csv"
OBJECTS_PATH = BASE_DIR / "objects_large_demo.csv"
REL_EXT_PATH = BASE_DIR / "relations_ext_large_demo.csv"
SUMMARY_PATH = BASE_DIR / "summary.json"

# =========================================================
# 1. 유틸
# =========================================================
def product_oid(lot_id, slot_no):
    if pd.isna(lot_id) or pd.isna(slot_no):
        return None
    return f"{str(lot_id)}_{int(slot_no):02d}"


def normalize_module_type(m):
    if pd.isna(m):
        return None
    m = str(m).upper().strip()

    if m.startswith(("DST", "PST", "TNS", "ORT", "LLM")):
        return m[:3]
    elif m.startswith(("PM", "LA", "TA", "LP")):
        return m[:2]
    else:
        return m


def _shorten(text: str, max_chars: int) -> str:
    if text is None:
        return ""
    return text if len(text) <= max_chars else text[:max_chars] + "\n... [TRUNCATED]"


def _table_lines(df: pd.DataFrame, cols, max_rows: int = 10) -> str:
    if df is None or len(df) == 0:
        return "- (none)"
    out = []
    for _, r in df.head(max_rows).iterrows():
        parts = []
        for c in cols:
            v = r.get(c, None)
            if isinstance(v, float):
                v = round(v, 2)
            parts.append(f"{c}={v}")
        out.append("- " + ", ".join(parts))
    return "\n".join(out)


def _iqr_anomaly_stats(dur_values: np.ndarray):
    if len(dur_values) == 0:
        return dict(avg=0.0, count=0, gt_1800=0, gt_1800_pct=0.0, iqr_upper=0.0, anomalies=0)
    avg = float(dur_values.mean())
    gt_1800 = int((dur_values > 1800).sum())
    gt_1800_pct = float(gt_1800 / len(dur_values) * 100)
    q1, q3 = np.percentile(dur_values, [25, 75])
    iqr = q3 - q1
    upper = float(q3 + 1.5 * iqr)
    anomalies = int((dur_values > upper).sum())
    return dict(avg=avg, count=len(dur_values), gt_1800=gt_1800, gt_1800_pct=gt_1800_pct, iqr_upper=upper, anomalies=anomalies)


# =========================================================
# 2. 네가 만든 PI Graph 생성 함수
#    (필요 최소 수정 없이 그대로 연결 가능하게 유지)
# =========================================================
import pm4py
import networkx as nx
from pm4py.objects.ocel.obj import OCEL
from collections import Counter

def build_full_pi_graph(raw_df: pd.DataFrame):
    df = raw_df.copy()
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])
    df["dur_s"] = (df["end_time"] - df["start_time"]).dt.total_seconds()

    id_vars = [
        'eqp_id', 'area', 'module_id', 'module_type',
        'srcmoduleid', 'srcmoduletype', 'dstmoduleid', 'dstmoduletype',
        'state', 'lot_id', 'slot_no', 'dur_s'
    ]

    melted = pd.melt(
        df,
        id_vars=id_vars,
        value_vars=['start_time', 'end_time'],
        var_name='time_type',
        value_name='ocel:timestamp'
    )

    melted["lifecycle"] = melted["time_type"].map({'start_time': 'start', 'end_time': 'complete'})
    melted["ocel:activity"] = melted["state"] + "_" + melted["time_type"].str.replace("_time", "").str.upper()
    melted["dur_s"] = melted.apply(lambda r: r["dur_s"] if r["lifecycle"] == "complete" else 0, axis=1)

    melted = melted.sort_values(['lot_id', 'slot_no', 'ocel:timestamp']).reset_index(drop=True)
    melted["ocel:eid"] = [f"e{i+1}" for i in range(len(melted))]

    events = melted[[
        "ocel:eid", "ocel:activity", "ocel:timestamp", "lifecycle", "dur_s",
        "lot_id", "slot_no", "eqp_id", "module_id",
        "srcmoduleid", "dstmoduleid", "state", "area"
    ]].copy()

    # Product
    prod = df.dropna(subset=["lot_id", "slot_no"]).copy()
    prod["ocel:oid"] = prod.apply(lambda r: product_oid(r["lot_id"], r["slot_no"]), axis=1)
    prod["ocel:type"] = "Product"
    prod = prod[["ocel:oid", "ocel:type", "lot_id", "slot_no"]].drop_duplicates()

    # FOUP
    foup = df[df["lot_id"].notna() & df["slot_no"].isna()].copy()
    foup["ocel:oid"] = foup["lot_id"].astype(str)
    foup["ocel:type"] = "FOUP"
    foup = foup[["ocel:oid", "ocel:type", "lot_id"]].drop_duplicates()

    # Equipment
    eqp = df[["eqp_id", "area"]].dropna(subset=["eqp_id"]).drop_duplicates()
    eqp["ocel:oid"] = eqp["eqp_id"]
    eqp["ocel:type"] = "Equipment"
    eqp = eqp[["ocel:oid", "ocel:type", "area"]]

    # Module
    module_cand = pd.concat([
        df[["module_id", "module_type"]].rename(columns={"module_id": "mid"}),
        df[["srcmoduleid", "srcmoduletype"]].rename(columns={"srcmoduleid": "mid", "srcmoduletype": "module_type"}),
        df[["dstmoduleid", "dstmoduletype"]].rename(columns={"dstmoduleid": "mid", "dstmoduletype": "module_type"})
    ], ignore_index=True)

    module_cand["module_type"] = module_cand["mid"].apply(normalize_module_type)

    mod = module_cand.dropna(subset=["mid"]).drop_duplicates(subset=["mid"])
    mod["ocel:oid"] = mod["mid"]
    mod["ocel:type"] = "Module"
    mod = mod[["ocel:oid", "ocel:type", "module_type"]]

    objects = pd.concat([prod, foup, eqp, mod], ignore_index=True)

    rel_rows = []
    for _, r in melted.iterrows():
        eid = r["ocel:eid"]
        if pd.notna(r["lot_id"]):
            if pd.notna(r["slot_no"]):
                rel_rows.append((eid, product_oid(r["lot_id"], r["slot_no"]), "HAS_PRODUCT"))
            else:
                rel_rows.append((eid, str(r["lot_id"]), "HAS_FOUP"))
        if pd.notna(r["eqp_id"]):
            rel_rows.append((eid, r["eqp_id"], "ON_EQUIPMENT"))
        if pd.notna(r["module_id"]):
            rel_rows.append((eid, r["module_id"], "USES_MODULE"))
        if pd.notna(r["srcmoduleid"]):
            rel_rows.append((eid, r["srcmoduleid"], "FROM_MODULE"))
        if pd.notna(r["dstmoduleid"]):
            rel_rows.append((eid, r["dstmoduleid"], "TO_MODULE"))

    relations_link = pd.DataFrame(rel_rows, columns=["ocel:eid", "ocel:oid", "ocel:qualifier"]).drop_duplicates()

    relations_ext = (
        relations_link
        .merge(events, on="ocel:eid", how="left")
        .merge(objects[["ocel:oid", "ocel:type"]], on="ocel:oid", how="left")
    )

    ocel = OCEL(events=events, objects=objects, relations=relations_ext)

    G = nx.MultiDiGraph(name="Semicon_PI_Graph")

    for _, row in objects.iterrows():
        G.add_node(row["ocel:oid"], **row.to_dict(), layer="OCDM")

    for _, row in events.iterrows():
        G.add_node(row["ocel:eid"], **row.to_dict(), layer="Event_Data", ocel_type="Event")

    for _, row in relations_link.iterrows():
        G.add_edge(
            row["ocel:eid"],
            row["ocel:oid"],
            relation=row["ocel:qualifier"],
            layer="Relationships"
        )

    semantic_layer = {
        "kpi": {
            "Product": {"cycle_target_sec": 1800},
            "Module": {"max_purge_sec": 300},
            "Equipment": {"oee_target": 85}
        },
        "rules": [
            {"condition": "lifecycle == 'complete' and dur_s > 1800", "action": "ALERT_Maintenance", "severity": "High"},
            {"condition": "state == 'PURGE' and dur_s > 300", "action": "RootCause_Analysis", "severity": "Critical"}
        ]
    }
    G.graph["semantic_layer"] = semantic_layer

    return {
        "ocel": ocel,
        "pi_graph": G,
        "semantic_layer": semantic_layer,
        "summary": {
            "events": len(events),
            "objects": len(objects),
            "edges": G.number_of_edges()
        }
    }


# =========================================================
# 3. compact context 생성
# =========================================================
def _build_topk_metrics_from_raw(raw_df: pd.DataFrame, top_k: int = 10):
    df = raw_df.copy()
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])
    df["dur_s"] = (df["end_time"] - df["start_time"]).dt.total_seconds()

    tr = df[df["srcmoduleid"].notna() & df["dstmoduleid"].notna()].copy()
    transfer_edges = (
        tr.groupby(["srcmoduleid", "dstmoduleid"], as_index=False)
          .agg(count=("dur_s", "size"),
               avg_s=("dur_s", "mean"),
               p95_s=("dur_s", lambda x: x.quantile(0.95)),
               p99_s=("dur_s", lambda x: x.quantile(0.99)))
          .sort_values(["p95_s", "count"], ascending=[False, False])
          .head(top_k)
    )

    hw = df[df["module_id"].notna()].copy()
    module_proc = (
        hw.groupby(["module_id", "module_type", "state"], as_index=False)
          .agg(count=("dur_s", "size"),
               avg_s=("dur_s", "mean"),
               p95_s=("dur_s", lambda x: x.quantile(0.95)))
          .sort_values(["p95_s", "count"], ascending=[False, False])
          .head(top_k)
    )

    common = df[df["lot_id"].isna() & df["slot_no"].isna()].copy()
    common_events = (
        common.groupby(["module_id", "module_type", "state"], as_index=False)
              .agg(count=("dur_s", "size"),
                   total_s=("dur_s", "sum"),
                   p95_s=("dur_s", lambda x: x.quantile(0.95)))
              .sort_values(["total_s", "count"], ascending=[False, False])
              .head(top_k)
    )

    df_prod = df.dropna(subset=["lot_id", "slot_no"]).copy()
    df_prod["product"] = df_prod.apply(lambda r: product_oid(r["lot_id"], r["slot_no"]), axis=1)

    def _stage_label(r):
        if pd.notna(r.get("srcmoduleid")) and pd.notna(r.get("dstmoduleid")):
            return f"{r['srcmoduleid']}→{r['dstmoduleid']}"
        if pd.notna(r.get("module_id")):
            return str(r["module_id"])
        return None

    df_prod["stage"] = df_prod.apply(_stage_label, axis=1)
    df_prod = df_prod.dropna(subset=["stage"]).sort_values(["product", "start_time", "end_time"])
    seq_by_prod = df_prod.groupby("product")["stage"].apply(list).to_dict()

    def _compress_seq(seq, max_len=12):
        if not seq:
            return []
        out = [seq[0]]
        for s in seq[1:]:
            if s != out[-1]:
                out.append(s)
        if len(out) <= max_len:
            return out
        return out[:6] + ["…"] + out[-5:]

    trace_samples = []
    for p, seq in list(seq_by_prod.items())[:8]:
        trace_samples.append({"product": p, "sequence": _compress_seq(seq)})

    return transfer_edges, module_proc, common_events, trace_samples


def _safe_ocdfg_topk(ocel, top_k: int = 5):
    try:
        ocdfg = pm4py.discover_ocdfg(ocel)
    except Exception as e:
        return {"available": False, "error": str(e), "top_freq": [], "top_perf": []}

    top_freq = []
    top_perf = []

    try:
        if isinstance(ocdfg, dict):
            freq = ocdfg.get("frequency", {})
            perf = ocdfg.get("performance", {})
            top_freq = list(freq.keys())[:top_k] if hasattr(freq, "keys") else []
            top_perf = list(perf.keys())[:top_k] if hasattr(perf, "keys") else []
        else:
            top_freq = [str(ocdfg)[:200]]
    except Exception:
        pass

    return {"available": True, "top_freq": top_freq, "top_perf": top_perf}


def create_llm_context_v2(pi_result: dict,
                          raw_df: pd.DataFrame,
                          top_k: int = 10,
                          max_chars: int = 16000):
    G = pi_result.get("pi_graph", None)
    sem = pi_result.get("semantic_layer", {}) or {}
    ocel = pi_result.get("ocel", None)

    node_types = {}
    if G is not None:
        node_types = Counter(d.get("ocel:type", d.get("ocel_type", "Unknown")) for _, d in G.nodes(data=True))

    dur_values = np.array([
        d.get("dur_s", 0)
        for _, d in (G.nodes(data=True) if G is not None else [])
        if d.get("lifecycle") == "complete" and d.get("dur_s", 0) > 0
    ], dtype=float)
    dur_stats = _iqr_anomaly_stats(dur_values)

    transfer_edges, module_proc, common_events, trace_samples = _build_topk_metrics_from_raw(raw_df, top_k=top_k)
    ocdfg_hint = _safe_ocdfg_topk(ocel, top_k=5) if ocel is not None else {"available": False}

    sem_kpi = sem.get("kpi", {})
    sem_rules = sem.get("rules", [])
    sem_rules_short = sem_rules[:5] if isinstance(sem_rules, list) else sem_rules

    parts = []
    parts.append("# Fab Process Intelligence - Compact Rich Context")
    parts.append("## 1) Snapshot")
    parts.append(f"- raw_df_rows: {len(raw_df):,}")
    parts.append(f"- pi_result_summary: {pi_result.get('summary', {})}")
    if node_types:
        parts.append(f"- object_type_distribution: {dict(node_types)}")

    parts.append("\n## 2) Duration(dur_s) statistics (complete events only)")
    parts.append(f"- count: {dur_stats['count']:,}")
    parts.append(f"- avg_dur_s: {dur_stats['avg']:.2f}")
    parts.append(f"- >1800s: {dur_stats['gt_1800']:,} ({dur_stats['gt_1800_pct']:.2f}%)")
    parts.append(f"- IQR_upper_bound: {dur_stats['iqr_upper']:.2f}")
    parts.append(f"- anomalies_over_IQR: {dur_stats['anomalies']:,}")

    parts.append("\n## 3) Top transfer bottlenecks (by p95)")
    parts.append(_table_lines(transfer_edges, ["srcmoduleid","dstmoduleid","count","avg_s","p95_s","p99_s"], max_rows=top_k))

    parts.append("\n## 4) Top module processing hotspots (by p95)")
    parts.append(_table_lines(module_proc, ["module_id","module_type","state","count","avg_s","p95_s"], max_rows=top_k))

    parts.append("\n## 5) Common events (no lot/slot) - resource availability signals")
    parts.append(_table_lines(common_events, ["module_id","module_type","state","count","total_s","p95_s"], max_rows=top_k))

    parts.append("\n## 6) Product trace backbone samples (compressed)")
    if trace_samples:
        for t in trace_samples[:8]:
            parts.append(f"- {t['product']}: " + " -> ".join(t["sequence"]))
    else:
        parts.append("- (none)")

    parts.append("\n## 7) Semantic layer (truncated)")
    parts.append("KPI targets:")
    parts.append(_shorten(json.dumps(sem_kpi, ensure_ascii=False, indent=2), 2000))
    parts.append("Rules (top 5):")
    parts.append(_shorten(json.dumps(sem_rules_short, ensure_ascii=False, indent=2), 2000))

    if ocdfg_hint.get("available"):
        parts.append("\n## 8) pm4py OCDFG hint (best-effort)")
        parts.append(f"- top_frequency_keys: {ocdfg_hint.get('top_freq', [])}")
        parts.append(f"- top_performance_keys: {ocdfg_hint.get('top_perf', [])}")
    else:
        if "error" in ocdfg_hint:
            parts.append("\n## 8) pm4py OCDFG hint")
            parts.append(f"- not available: {ocdfg_hint['error']}")

    parts.append("\n## RULES_FOR_LLM")
    parts.append("- Use ONLY this context.")
    parts.append("- Separate facts vs hypotheses.")
    parts.append("- If more data is needed, request a specific slice: (module/edge/product, time window, metric).")
    parts.append("- Do NOT propose actions outside the allowed action space unless explicitly provided.")

    return _shorten("\n".join(parts), max_chars)


# =========================================================
# 4. 프롬프트 생성
# =========================================================
def build_daily_analysis_prompt(ctx: str) -> str:
    return f"""SYSTEM
You are a semiconductor fab process intelligence analyst.
Your role is to extract valid operational insights from a compact PI-Graph snapshot.

Non-negotiable rules:
1) Use ONLY the provided CONTEXT.
2) Distinguish FACT vs HYPOTHESIS explicitly.
3) For every insight, cite the supporting evidence by quoting the exact line/item from the context.
4) Do NOT produce generic advice. Tie every statement to specific modules, edges, states, or metrics.
5) If something is missing, say: "Not derivable from the context."

USER
CONTEXT:
<<<
{ctx}
>>>

TASK
Produce a high-value daily summary for fab operations.

OUTPUT FORMAT (STRICT)

A) Executive Summary (max 6 bullets)
- Each bullet must include [FACT] or [HYPOTHESIS]
- Each bullet must include a short evidence quote

B) Bottleneck Candidates (top 5)
For each:
1. Candidate
2. Type: Transfer / Processing / Resource-availability
3. Evidence quote
4. Why it matters
5. Next check using existing data only

C) Common-event Impact Analysis
- Explain whether common events such as PURGE or CHAMBER_CLEAN could affect transfer or processing delays
- Provide 2 mechanisms
- Mark both as [HYPOTHESIS]
- If unsupported, say "Not derivable from the context."

D) Immediate Experiments (exactly 3)
For each:
1. Goal
2. Procedure (3–5 steps)
3. Success metric (avg_s / p95_s / p99_s / total_s / count / anomaly_rate)
4. Expected outcome
5. Risk / side effect

E) Additional Data Requests (optional, max 5)
For each:
- slice_target
- time_window
- metric(s)
- rationale

QUALITY CHECK
- Every numeric statement must appear in the context.
- Every recommendation must be linked to a bottleneck candidate.
- If uncertain, reduce certainty and say why.
"""


# =========================================================
# 5. 사내 LLM 호출부 placeholder
# =========================================================
def call_internal_llm(prompt: str) -> str:
    """
    여기를 네 사내 LLM API 호출 코드로 교체.
    예:
    - requests.post(...)
    - internal_sdk.chat(...)
    """
    return "LLM API 호출부를 여기에 연결하세요."


# =========================================================
# 6. 엔드투엔드 실행
# =========================================================
def run_validation_pipeline(raw_path=RAW_PATH,
                            save_prompt_path="daily_prompt.txt",
                            save_ctx_path="daily_context.txt",
                            max_ctx_chars=16000,
                            top_k=10):
    raw_df = pd.read_csv(raw_path, parse_dates=["start_time", "end_time"])

    print(f"[INFO] raw_df loaded: {raw_df.shape}")

    pi_result = build_full_pi_graph(raw_df)
    print(f"[INFO] PI Graph built: {pi_result['summary']}")

    ctx = create_llm_context_v2(
        pi_result=pi_result,
        raw_df=raw_df,
        top_k=top_k,
        max_chars=max_ctx_chars
    )
    print(f"[INFO] context length: {len(ctx):,} chars")

    prompt = build_daily_analysis_prompt(ctx)
    print(f"[INFO] prompt length: {len(prompt):,} chars")

    with open(save_ctx_path, "w", encoding="utf-8") as f:
        f.write(ctx)

    with open(save_prompt_path, "w", encoding="utf-8") as f:
        f.write(prompt)

    return {
        "raw_df": raw_df,
        "pi_result": pi_result,
        "context": ctx,
        "prompt": prompt
    }


# =========================================================
# 7. 선택: OCPM 시각화도 같이 확인
# =========================================================
def run_ocpm_visualization(pi_result,
                           save_ocdfg_freq="ocdfg_freq.png",
                           save_ocdfg_perf="ocdfg_perf.png",
                           save_ocpn="ocpn.png"):
    ocel = pi_result["ocel"]

    ocdfg = pm4py.discover_ocdfg(ocel)
    pm4py.save_vis_ocdfg(ocdfg, save_ocdfg_freq, annotation="frequency")
    pm4py.save_vis_ocdfg(ocdfg, save_ocdfg_perf, annotation="performance")

    ocpn = pm4py.discover_oc_petri_net(ocel)
    pm4py.save_vis_ocpn(ocpn, save_ocpn)

    print("[INFO] saved:", save_ocdfg_freq, save_ocdfg_perf, save_ocpn)


# =========================================================
# 8. 실행 예시
# =========================================================
if __name__ == "__main__":
    result = run_validation_pipeline(
        raw_path=RAW_PATH,
        save_prompt_path="daily_prompt.txt",
        save_ctx_path="daily_context.txt",
        max_ctx_chars=16000,
        top_k=10
    )

    # 필요 시 시각화도 실행
    # run_ocpm_visualization(result["pi_result"])

    # 사내 LLM 붙일 경우
    # llm_output = call_internal_llm(result["prompt"])
    # print(llm_output)

FileNotFoundError: [Errno 2] No such file or directory: '..logs\\raw_df_large_demo.csv'